# Feature Extraction from Labeled Frame Folders
This notebook extracts pose and movement features from labeled frame folders stored under `data/sorted_frames_*`.
It is designed to produce a clean CSV dataset from the current project assets.

In [1]:
from src.extraction import extract_dataset_from_frame_root, load_label_map
import os

# Define the frame dataset folders and their corresponding device names
FRAME_FOLDERS = [
    ('data/sorted_frames_a13', 'a13'),
    ('data/sorted_frames_note9', 'note9'),
    ('data/sorted_frames_s10e', 's10e')
]

OUTPUT_CSV = 'data/extracted_video_data.csv'

label_map = load_label_map()
print('Label mapping loaded:', label_map)
print('Frame folders to process:')
for folder, device in FRAME_FOLDERS:
    exists = os.path.exists(folder)
    print(f'  - {folder} (device: {device}) - {"EXISTS" if exists else "NOT FOUND"}')


2026-05-26 22:42:34.883084: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-26 22:42:34.888383: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-26 22:42:34.902137: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-26 22:42:34.928596: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-26 22:42:34.928650: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-26 22:42:34.945731: I tensorflow/core/platform/cpu_feature_guard.cc:

Label mapping loaded: {'closed_guard': 'ground', 'open_guard': 'ground', 'pin': 'ground', 'submission': 'ground', 'front_kick': 'kick', 'head_kick': 'kick', 'leg_kick': 'kick', 'mid_kick': 'kick', 'body_hook': 'punch', 'jab': 'punch', 'lead_hook': 'punch', 'rear_hook': 'punch', 'straight': 'punch', 'double_leg': 'takedown', 'single_leg': 'takedown', 'sprawl': 'takedown', 'not_engaged': 'not_engaged'}
Frame folders to process:
  - data/sorted_frames_a13 (device: a13) - EXISTS
  - data/sorted_frames_note9 (device: note9) - EXISTS
  - data/sorted_frames_s10e (device: s10e) - EXISTS


In [2]:
import csv
from pathlib import Path
from src.extraction import extract_frame_features, build_feature_row, get_feature_header, load_ssd_model

# Load the SSD model once for reuse across all folders
ssd_model = load_ssd_model()
print('Loaded SSD detection model')

# Initialize output CSV with header
output_file = Path(OUTPUT_CSV)
output_file.parent.mkdir(parents=True, exist_ok=True)

total_count = 0

with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(get_feature_header())
    
    # Process each device folder
    for source_root, device_name in FRAME_FOLDERS:
        if not os.path.exists(source_root):
            print(f'Skipping {source_root} - folder not found')
            continue
        
        print(f'\nProcessing {device_name} from {source_root}...')
        device_count = 0
        
        # Walk through all subdirectories (labels)
        for root, dirs, files in os.walk(source_root):
            image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
            if not image_files:
                continue
            
            raw_label = os.path.basename(root)
            simplified_label = label_map.get(raw_label, raw_label)
            image_paths = sorted([os.path.join(root, f) for f in image_files])
            
            for image_path in image_paths:
                frame_id = Path(image_path).stem
                result = extract_frame_features(image_path, ssd_model)
                if result is None:
                    continue
                
                landmarks1, landmarks2 = result
                row = build_feature_row(landmarks1, landmarks2, frame_id, raw_label, simplified_label, device_name)
                if row is not None:
                    writer.writerow(row)
                    device_count += 1
                    total_count += 1
        
        print(f'  Extracted {device_count} feature rows from {device_name}')

print(f'\n✓ COMPLETE: Extracted {total_count} total feature rows to {OUTPUT_CSV}')


Loaded SSD detection model

Processing a13 from data/sorted_frames_a13...


I0000 00:00:1779860573.471306  352252 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779860573.478322  352383 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.0+git2605062153.8d005e3d666~n~mesarc3), renderer: AMD Radeon Vega 8 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-29-generic)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
/home/garymon/Downloads/CapstoneProject/projectFolder/updated_workflow/.venv/lib/python3.12/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
I0000 00:00:1779860573.753369  352252 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779860573.757015  352395 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.0+git2605062153.8d005

  Extracted 3007 feature rows from a13

Processing note9 from data/sorted_frames_note9...


I0000 00:00:1779863115.891015  352252 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779863115.897621  454537 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.0+git2605062153.8d005e3d666~n~mesarc3), renderer: AMD Radeon Vega 8 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-29-generic)
I0000 00:00:1779863116.257796  352252 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779863116.261400  454549 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.0+git2605062153.8d005e3d666~n~mesarc3), renderer: AMD Radeon Vega 8 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-29-generic)
I0000 00:00:1779863116.630466  352252 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779863116.635254  454561 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.0+git2605062153.8d005e3d666~n~mesarc3), renderer: AMD Radeon Vega 8 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-29-g

  Extracted 3224 feature rows from note9

Processing s10e from data/sorted_frames_s10e...


I0000 00:00:1779865576.961629  352252 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779865576.965948  554434 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.0+git2605062153.8d005e3d666~n~mesarc3), renderer: AMD Radeon Vega 8 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-29-generic)
I0000 00:00:1779865577.160745  352252 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779865577.175479  554453 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.0+git2605062153.8d005e3d666~n~mesarc3), renderer: AMD Radeon Vega 8 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-29-generic)
I0000 00:00:1779865577.820417  352252 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779865577.844552  554466 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.0+git2605062153.8d005e3d666~n~mesarc3), renderer: AMD Radeon Vega 8 Graphics (radeonsi, raven, ACO, DRM 3.64, 6.17.0-29-g

  Extracted 3332 feature rows from s10e

✓ COMPLETE: Extracted 9563 total feature rows to data/extracted_video_data.csv


## Verify and Clean Extracted Features
This section loads the extracted CSV output, checks for missing values, and saves a cleaned dataset copy for modeling.

In [ ]:
import pandas as pd

VIDEO_CSV = OUTPUT_CSV
CLEANED_CSV = 'data/extracted_video_data_clean.csv'

print('Loading extracted feature CSV:', VIDEO_CSV)
df = pd.read_csv(VIDEO_CSV)
print('Original shape:', df.shape)
print('Missing values per column:')
print(df.isna().sum())
print('\nLabel distribution:')
print(df['Label'].value_counts().head(20))

# Save a cleaned dataset copy with any incomplete rows removed.
df_clean = df.dropna().reset_index(drop=True)
df_clean.to_csv(CLEANED_CSV, index=False)
print('\nSaved cleaned dataset:', CLEANED_CSV)
print('Clean shape:', df_clean.shape)

## Notes
- The extraction function detects two people in each frame and computes paired joint coordinates, angle features, and pose differences.
- If a frame does not contain two detected persons, it is skipped.
- The output CSV includes `Label` and `Label_Simplified` columns based on `data/class_dictionary.txt`.